# Aircraft Trajectory Reconstruction -- Demo

**Project:** Reconstructing missing ADS-B trajectories over the North Atlantic Ocean

---

### The Problem

Commercial aircraft broadcast their position via **ADS-B** every few seconds. Over oceans, ground stations lose coverage, creating gaps of **3-5 hours** in the trajectory data.

We use sparse **ADS-C** waypoints (satellite-reported every 10-30 min) as ground truth for evaluation -- never shown to any reconstruction method.

### Three Reconstruction Methods

| Method | Type | Key idea |
|--------|------|----------|
| **Baseline** | Geometric | Great-circle arc between gap endpoints |
| **Kalman** | Physics | Rauch-Tung-Striebel smoother anchored at both ends |
| **GRU** | Deep Learning | Sequence model trained on 1540 historical flights |

**Evaluation metric:** Mean nearest-neighbour haversine distance from each ADS-C waypoint to the nearest reconstructed point (km).

**Dataset split:** 1540 train / 315 val / **291 test** -- the interactive explorer shows only the held-out test flights.

## Step 1 -- Setup

Run this cell once. Loads libraries, GRU model, pre-computed errors, and defines all helper functions.

In [27]:
import sys, os, json as _json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output

NB_DIR   = Path(r'C:\Users\marko\Desktop\AeroML3(og)\AeroML3\notebook')
ROOT_DIR = NB_DIR.parent
sys.path.insert(0, str(NB_DIR))

NOEL_DIR  = ROOT_DIR / 'noel_part'
CLEAN_DIR = NOEL_DIR / 'cleaned_data_final'
OUT_DIR   = ROOT_DIR / 'outputs'
OUT_DIR.mkdir(exist_ok=True)
os.chdir(NOEL_DIR)
sys.path.insert(0, str(NOEL_DIR))

SPLIT_PATH = ROOT_DIR / 'artifacts' / 'step4_ml_dataset' / 'catalog' / 'flight_splits.parquet'
EVAL_PATH  = ROOT_DIR / 'outputs' / 'evaluation_results.csv'
SEQ_PATH   = ROOT_DIR / 'artifacts' / 'step4_ml_dataset' / 'dataset' / 'sequences_test.npz'
MODEL_PATH = ROOT_DIR / 'step5_v2' / 'best_model_v2.pt'
PRED_PATH  = ROOT_DIR / 'step5_v2' / 'test_predictions.npz'

from step3_baseline       import reconstruct_gap_baseline
from step5_kalman_aeroml3 import reconstruct_single_kalman
from step5_train_gru_v2   import TrajectoryGRU, gc_interpolate_batch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_gru = TrajectoryGRU().to(device)
model_gru.load_state_dict(torch.load(str(MODEL_PATH), map_location=device))
model_gru.eval()

# Pre-computed test sequences (exact arrays used during training)
_npz = np.load(str(SEQ_PATH), allow_pickle=True)
_seg_ids       = _npz['segment_ids']
_flight_to_idx = {sid.split('/')[-1]: i for i, sid in enumerate(_seg_ids)}

# Load test split catalog and pre-computed errors
splits_df  = pd.read_parquet(SPLIT_PATH)
eval_df    = pd.read_csv(EVAL_PATH).set_index('flight')
test_split = splits_df[splits_df['split'] == 'test'].reset_index(drop=True)

test_flights = []
for _, row in test_split.iterrows():
    fd   = CLEAN_DIR / row['source_run'] / row['flight_name']
    name = row['flight_name']
    if fd.exists() and name in eval_df.index and name in _flight_to_idx:
        test_flights.append((name, fd))

flight_map = {name: fd for name, fd in test_flights}

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def haversine_km(lat1, lon1, lat2, lon2):
    """Vectorised haversine distance (km)."""
    R  = 6371.0
    p1 = np.radians(lat1); p2 = np.radians(lat2)
    dp = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a  = np.sin(dp/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def _smooth_jerk(lats, lons, n_iter=6, alpha=0.35):
    """Laplacian smoothing that reduces 3rd-order jerk at gap boundaries.

    Each interior point is blended alpha-way toward the average of its two
    neighbours.  Endpoints are NEVER moved, so the hard clamp is preserved.
    More iterations = smoother entry/exit curve from the ADS-B track.
    """
    lats, lons = lats.copy(), lons.copy()
    if len(lats) <= 2:
        return lats, lons
    for _ in range(n_iter):
        avg_lat = 0.5 * (lats[:-2] + lats[2:])
        avg_lon = 0.5 * (lons[:-2] + lons[2:])
        lats[1:-1] = (1.0 - alpha) * lats[1:-1] + alpha * avg_lat
        lons[1:-1] = (1.0 - alpha) * lons[1:-1] + alpha * avg_lon
    return lats, lons


# ---------------------------------------------------------------------------
# GRU inference with hard endpoint constraint + jerk smoothing
# ---------------------------------------------------------------------------

def run_gru_inference(flight_name, t0, t1, dt=15.0):
    """Predict the gap trajectory.

    Endpoint constraint (the handshake):
      Step 1 – sin(pi*tau) blending pulls every point toward the great-circle
               baseline.  The baseline equals the before-anchor at tau=0 and
               the after-anchor at tau=1, so blending weight → 0 near both
               ends.
      Step 2 – Hard clamp: pred[0]  = before_anchor  (exact, no rounding)
                           pred[-1] = after_anchor   (exact, no rounding)
      Step 3 – _smooth_jerk: Laplacian pass preserving both clamped endpoints,
               smooths the tangent angle so there is no sharp kink where the
               predicted path joins the ADS-B track.
    """
    idx   = _flight_to_idx[flight_name]
    gap_s = float(_npz['gap_dur_sec'][idx])
    n_out = max(1, int(round(gap_s / dt)))
    taus  = np.array([(i + 1) / (n_out + 1) for i in range(n_out)], dtype=np.float32)

    la0 = _npz['before_anchor_lat'][idx:idx+1].astype(np.float32)
    lo0 = _npz['before_anchor_lon'][idx:idx+1].astype(np.float32)
    la1 = _npz['after_anchor_lat'][idx:idx+1].astype(np.float32)
    lo1 = _npz['after_anchor_lon'][idx:idx+1].astype(np.float32)
    bl_lat, bl_lon = gc_interpolate_batch(la0, lo0, la1, lo1, taus[None, :])
    bl_lat_1d = bl_lat[0]
    bl_lon_1d = bl_lon[0]

    batch = {
        'before_seq':   torch.FloatTensor(_npz['before_seq'][idx:idx+1]).to(device),
        'before_mask':  torch.FloatTensor(_npz['before_mask'][idx:idx+1]).to(device),
        'after_seq':    torch.FloatTensor(_npz['after_seq'][idx:idx+1]).to(device),
        'after_mask':   torch.FloatTensor(_npz['after_mask'][idx:idx+1]).to(device),
        'gap_norm':     torch.FloatTensor([gap_s / 6000.0]).to(device),
        'adsc_tau':     torch.FloatTensor(taus[None]).to(device),
        'baseline_lat': torch.FloatTensor(bl_lat.astype(np.float32)).to(device),
        'baseline_lon': torch.FloatTensor(bl_lon.astype(np.float32)).to(device),
    }
    with torch.no_grad():
        pred_lat, pred_lon = model_gru(batch)
    pred_lat = pred_lat.cpu().numpy()[0]
    pred_lon = pred_lon.cpu().numpy()[0]

    # Step 1 – sin(pi*tau) blending toward baseline near endpoints
    w        = np.sin(np.pi * taus)
    pred_lat = w * pred_lat + (1.0 - w) * bl_lat_1d
    pred_lon = w * pred_lon + (1.0 - w) * bl_lon_1d

    # Step 2 – hard clamp: mathematically force exact anchor coords
    pred_lat[0]  = float(la0[0]);  pred_lat[-1]  = float(la1[0])
    pred_lon[0]  = float(lo0[0]);  pred_lon[-1]  = float(lo1[0])

    # Step 3 – jerk smoothing (endpoints preserved by _smooth_jerk)
    pred_lat, pred_lon = _smooth_jerk(pred_lat, pred_lon)

    return pd.DataFrame({
        'timestamp': [t0 + pd.Timedelta(seconds=dt * (i + 1)) for i in range(n_out)],
        'latitude':  pred_lat,
        'longitude': pred_lon,
    })


print(f'Device       : {device}')
print(f'GRU model    : {MODEL_PATH.name}')
print(f'Test flights : {len(test_flights)} (held-out, never seen during training)')
print('Setup complete.')


Device       : cpu
GRU model    : best_model_v2.pt
Test flights : 291 (held-out, never seen during training)
Setup complete.


## Step 2 -- Interactive Flight Explorer

All **291 held-out test flights** are available in the dropdown.

- **Map (left):** all four tracks overlaid -- grey=ADS-B, red=Baseline, blue=Kalman, green=GRU, gold=ADS-C truth
- **Bar chart (right):** pre-computed errors from the full evaluation run (same numbers as the aggregate results)
- Gold border = best method on this flight

> Run the **Setup** cell first, then click **Load & Reconstruct**.

In [28]:
flight_names = [name for name, _ in test_flights]

dropdown = widgets.Dropdown(
    options=flight_names,
    value=flight_names[0],
    description='Flight:',
    layout=widgets.Layout(width='560px')
)
btn = widgets.Button(
    description='Load & Reconstruct',
    button_style='primary',
    icon='plane',
    layout=widgets.Layout(width='195px', height='32px')
)
status = widgets.Label(value='Ready.')
out    = widgets.Output()

def show_flight(flight_name):
    fd = flight_map[flight_name]

    # -- Load data -------------------------------------------------------
    bef  = pd.read_parquet(fd / 'adsb_before.parquet')
    aft  = pd.read_parquet(fd / 'adsb_after.parquet')
    adsc = pd.read_parquet(fd / 'adsc.parquet')
    for df in [bef, aft, adsc]:
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True).dt.tz_localize(None)
        df.sort_values('timestamp', inplace=True)
        df.reset_index(drop=True, inplace=True)

    t0 = bef['timestamp'].iloc[-1]
    t1 = aft['timestamp'].iloc[0]
    gap_min  = (t1 - t0).total_seconds() / 60
    adsc_gap = adsc[(adsc['timestamp'] > t0) & (adsc['timestamp'] < t1)].reset_index(drop=True)
    bef_trim = bef[bef['timestamp'] >= t0 - pd.Timedelta(minutes=30)].reset_index(drop=True)
    aft_trim = aft[aft['timestamp'] <= t1 + pd.Timedelta(minutes=30)].reset_index(drop=True)

    # -- Run reconstructions for the map ---------------------------------
    rb = reconstruct_gap_baseline(bef_trim, aft_trim)
    rk = reconstruct_single_kalman(bef_trim, aft_trim)
    rg = run_gru_inference(flight_name, t0, t1)

    # -- Use pre-computed errors from evaluation run ---------------------
    row = eval_df.loc[flight_name]
    eb  = float(row['baseline_err_km'])
    ek  = float(row['kalman_err_km'])
    eg  = float(row['gru_err_km'])

    # -- Figure ----------------------------------------------------------
    fig = plt.figure(figsize=(18, 8))
    gs  = gridspec.GridSpec(1, 2, width_ratios=[3, 1], wspace=0.28)

    # Left: trajectory map
    ax_map = fig.add_subplot(gs[0])
    ax_map.plot(bef['longitude'], bef['latitude'],
                color='#888888', lw=1.5, label='ADS-B before')
    ax_map.plot(aft['longitude'], aft['latitude'],
                color='#888888', lw=1.5, ls=':', label='ADS-B after')
    ep_lat0 = float(bef_trim['latitude'].iloc[-1])
    ep_lon0 = float(bef_trim['longitude'].iloc[-1])
    ep_lat1 = float(aft_trim['latitude'].iloc[0])
    ep_lon1 = float(aft_trim['longitude'].iloc[0])
    def _connected_lats(r): return np.concatenate([[ep_lat0], r['latitude'].values,  [ep_lat1]])
    def _connected_lons(r): return np.concatenate([[ep_lon0], r['longitude'].values, [ep_lon1]])
    ax_map.plot(_connected_lons(rb), _connected_lats(rb),
                color='#F44336', lw=2.0, ls='--', label=f'Baseline  {eb:.1f} km')
    ax_map.plot(_connected_lons(rk), _connected_lats(rk),
                color='#2196F3', lw=2.0,            label=f'Kalman    {ek:.1f} km')
    ax_map.plot(_connected_lons(rg), _connected_lats(rg),
                color='#4CAF50', lw=2.5,            label=f'GRU       {eg:.1f} km')
    if len(adsc_gap) > 0:
        ax_map.plot(adsc_gap['longitude'], adsc_gap['latitude'],
                    color='#FFC107', lw=3.0, marker='o', ms=6,
                    label='ADS-C truth (ground truth)')
    ax_map.scatter(float(bef_trim['longitude'].iloc[-1]),
                   float(bef_trim['latitude'].iloc[-1]),
                   s=140, color='#1a1a1a', zorder=6, marker='^')
    ax_map.scatter(float(aft_trim['longitude'].iloc[0]),
                   float(aft_trim['latitude'].iloc[0]),
                   s=140, color='#1a1a1a', zorder=6, marker='v')
    ax_map.set_xlabel('Longitude', fontsize=12)
    ax_map.set_ylabel('Latitude',  fontsize=12)
    ax_map.set_title(
        f'{flight_name}\n'
        f'Gap: {gap_min:.0f} min  |  ADS-C waypoints: {len(adsc_gap)}  |  '
        f'Test flight #{flight_names.index(flight_name)+1} of {len(flight_names)}',
        fontsize=11, fontweight='bold'
    )
    ax_map.legend(fontsize=10, loc='best', framealpha=0.9)
    ax_map.grid(True, alpha=0.3)

    # Right: error bar chart
    ax_bar = fig.add_subplot(gs[1])
    methods = ['Baseline', 'Kalman', 'GRU']
    errors  = [eb, ek, eg]
    colors  = ['#F44336', '#2196F3', '#4CAF50']
    best_i  = int(np.argmin(errors))
    bars = ax_bar.barh(methods, errors, color=colors, edgecolor='white', height=0.5)
    bars[best_i].set_edgecolor('#FFD700')
    bars[best_i].set_linewidth(3)
    for bar, val in zip(bars, errors):
        ax_bar.text(val + max(errors)*0.01, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f} km', va='center', fontsize=11, fontweight='bold')
    ax_bar.set_xlabel('Mean error vs ADS-C (km)', fontsize=11)
    ax_bar.set_title('Accuracy\n(lower is better, gold = best)', fontsize=11, fontweight='bold')
    ax_bar.set_xlim(0, max(errors) * 1.40)
    ax_bar.grid(axis='x', alpha=0.3)
    ax_bar.invert_yaxis()

    fig.suptitle('Trajectory Reconstruction -- Test Set Evaluation',
                 fontsize=14, fontweight='bold')
    # rect=[left, bottom, right, top] reserves space at top for suptitle
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    print(f'Best method : {methods[best_i]}  '
          f'({(1 - errors[best_i]/eb)*100:.1f}% better than baseline)')

def on_click(b):
    status.value = 'Reconstructing...'
    btn.disabled = True
    with out:
        clear_output(wait=True)
        try:
            show_flight(dropdown.value)
        except Exception as e:
            import traceback
            traceback.print_exc()
    status.value = 'Done.'
    btn.disabled = False

btn.on_click(on_click)
display(widgets.HBox([dropdown, btn, status]), out)
on_click(None)  # auto-run first flight


Output()

## Step 3 -- Aggregate Results (Test Set)

Overall performance across all 291 test flights.

Run this cell to see the summary table and distributions.

In [ ]:
# ── Filter to test split ────────────────────────────────────────────────────
test_names = [name for name, _ in test_flights]
test_eval  = eval_df.loc[eval_df.index.intersection(test_names)].copy()

print('=' * 60)
print('AGGREGATE RESULTS  (pre-computed, test set)')
print('=' * 60)

summary = pd.DataFrame({
    'Method':            ['Baseline', 'Kalman', 'GRU'],
    'Mean Error (km)':   [round(test_eval['baseline_err_km'].mean(), 1),
                          round(test_eval['kalman_err_km'].mean(),   1),
                          round(test_eval['gru_err_km'].mean(),      1)],
    'Median Error (km)': [round(test_eval['baseline_err_km'].median(), 1),
                          round(test_eval['kalman_err_km'].median(),   1),
                          round(test_eval['gru_err_km'].median(),      1)],
})
base_mean = summary.loc[0, 'Mean Error (km)']
summary['vs Baseline'] = summary['Mean Error (km)'].apply(
    lambda x: '--' if x == base_mean else f'{(1 - x/base_mean)*100:+.1f}%'
)
display(summary.style
    .highlight_min(subset=['Mean Error (km)', 'Median Error (km)'], color='#c8f7c5')
    .set_properties(**{'text-align': 'center', 'font-size': '13px'})
    .set_caption('Test Set Reconstruction Accuracy (lower is better)')
)

# ── Distribution plot (constrained_layout avoids tight_layout warning) ───────
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False,
                         constrained_layout=True)
pairs = [
    ('Baseline', 'baseline_err_km', '#F44336'),
    ('Kalman',   'kalman_err_km',   '#2196F3'),
    ('GRU',      'gru_err_km',      '#4CAF50'),
]
for ax, (name, col, color) in zip(axes, pairs):
    vals = test_eval[col].dropna()
    ax.hist(vals, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(vals.mean(),   color='black',   lw=2, ls='--', label=f'Mean   {vals.mean():.1f} km')
    ax.axvline(vals.median(), color='#FFD700', lw=2, ls='-',  label=f'Median {vals.median():.1f} km')
    ax.set_title(name, fontsize=13, fontweight='bold', color=color)
    ax.set_xlabel('Error (km)', fontsize=11)
    ax.set_ylabel('Flights',    fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
fig.suptitle('Error Distribution -- Test Set', fontsize=13, fontweight='bold')
plt.savefig(OUT_DIR / 'test_error_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════
# STRICT MASKED EVALUATION
# Only counts ADS-C waypoints where adsc_mask == 1 (inside the gap, non-padding).
# ═══════════════════════════════════════════════════════════════════════════
print()
print('=' * 60)
print('STRICT MASKED EVALUATION  (adsc_mask == 1 only)')
print('=' * 60)

if PRED_PATH.exists():
    pnpz = np.load(str(PRED_PATH))
    ge   = pnpz['gru_errors_m']        # [N, K]  haversine metres per tau slot
    be   = pnpz['baseline_errors_m']   # [N, K]
    M    = pnpz['mask']                # [N, K]  1 = real ADS-C waypoint inside gap

    gru_pf, bl_pf = [], []
    for i in range(len(ge)):
        valid = M[i] > 0
        if valid.sum() > 0:
            gru_pf.append(ge[i][valid].mean() / 1000.0)
            bl_pf.append(be[i][valid].mean()  / 1000.0)
    gru_pf = np.array(gru_pf)
    bl_pf  = np.array(bl_pf)

    imp_mean   = (1 - gru_pf.mean()     / bl_pf.mean())     * 100
    imp_median = (1 - np.median(gru_pf) / np.median(bl_pf)) * 100

    masked_summary = pd.DataFrame({
        'Method':            ['Baseline (masked)', 'GRU (masked)'],
        'Mean Error (km)':   [round(bl_pf.mean(),             2), round(gru_pf.mean(),             2)],
        'Median Error (km)': [round(float(np.median(bl_pf)),  2), round(float(np.median(gru_pf)),  2)],
        'P90 Error (km)':    [round(float(np.percentile(bl_pf,  90)), 2),
                              round(float(np.percentile(gru_pf, 90)), 2)],
    })
    display(masked_summary.style
        .highlight_min(subset=['Mean Error (km)', 'Median Error (km)'], color='#c8f7c5')
        .set_properties(**{'text-align': 'center', 'font-size': '13px'})
        .set_caption('Strict masked evaluation — only real ADS-C waypoints inside the gap')
    )
    print(f'  Flights evaluated : {len(gru_pf)}')
    print(f'  GRU improvement   : {imp_mean:.1f}% (mean)   {imp_median:.1f}% (median) vs Baseline')
else:
    print('  test_predictions.npz not found — run step5_train_gru_v2.py to generate it.')

# ═══════════════════════════════════════════════════════════════════════════
# PLAIN ENGLISH SUMMARY  — for a non-technical pilot
# ═══════════════════════════════════════════════════════════════════════════
print()
print('=' * 60)
print('WHAT DID THE MODEL ACTUALLY DO?  (Plain English)')
print('=' * 60)
print("""
Imagine you are flying from New York to London.  For roughly 3-4 hours
over the North Atlantic Ocean your GPS broadcast goes dark — ground
stations cannot pick up your signal.  We call that the "gap".

What we DO know:
  * Your exact position just BEFORE the gap started
    (the last ADS-B ping before you flew out of radar range).
  * Your exact position just AFTER the gap ended
    (the first ADS-B ping when you came back into radar range).
  * A handful of optional "check-in" satellite pings called ADS-C,
    which the aircraft sends roughly every 10-30 minutes during the gap.
    These are our ground truth — used ONLY to grade ourselves,
    never to help the reconstruction.

What each method tried to do:
  - Baseline  Draw the shortest curved line (great-circle arc) between
              the two known endpoints.  Fast, simple, no learning.

  - Kalman    Use physics: assume the plane kept roughly the same speed
              and heading, then stitch the two endpoints together with a
              smooth physics-based curve.  No training data needed.

  - GRU       A neural network trained on 1,540 historical North Atlantic
              flights.  It studied the last ~64 minutes of radar track
              before the gap and the first ~32 minutes after, then asked:
              "Given how THIS plane was flying, where did similar planes
              go in the past?"  The predicted path is mathematically forced
              to start exactly at the last known radar point and end exactly
              at the first radar point after the gap — no jumps, ever.

How we measured the error:
  For each satellite check-in ping (ADS-C waypoint) that fell INSIDE
  the gap we measured the straight-line distance between:

      WHERE WE PREDICTED the plane was  vs.  WHERE IT ACTUALLY WAS.

  We averaged those distances across all 291 test flights that the model
  had never seen before.  The result: "On average, our prediction was
  X km away from where the plane really was."  Lower = better.
  A perfect reconstruction would score 0 km.

  Only real, non-padded ADS-C waypoints strictly inside the gap were
  counted (adsc_mask == 1).  Padding zeros and out-of-gap points were
  completely ignored, so the number is honest.
""")


## Summary

### Key takeaways

- **Baseline** is fast and simple but ignores flight dynamics (jet streams, pressure gradients, curved great-circle routes).
- **Kalman** achieves a strong improvement (~27% better than baseline) using only physics -- no training data needed. It models velocity and anchors the trajectory at both gap endpoints.
- **GRU** learns route patterns from 1540 historical North Atlantic flights. It performs similarly to Kalman overall but can capture non-linear deviations that the Kalman filter cannot model.

### Why reconstruction quality matters

Without reconstruction, raw ADS-B **underestimates** the true route distance by ~400 km per oceanic flight:

- ~7,600 kg of CO2 per flight unaccounted for in emissions reporting
- Gaps in safety monitoring for oceanic airspace
- Inaccurate route analytics and fuel burn estimates

Better reconstruction closes this tracking gap.